# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs, referencing all entities by their `@id` fields.

In [ ]:
# Display all record sets and their associated fields and columns by @id
print("Available record sets (by @id):")
ds_record_sets = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in ds_record_sets]
for rs in ds_record_sets:
    print(f"- RecordSet @id: {rs['@id']}  (name: {rs.get('name','N/A')})")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - {field_id}")
    if 'column' in rs:
        print("  Columns:")
        for col in rs['column']:
            col_id = col['@id'] if isinstance(col, dict) else col
            print(f"    - {col_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration, select the first available record set
record_sets = record_set_ids
dataframes = {}

for record_set in record_sets:
    # Load records for this record set
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded DataFrame for RecordSet @id: {record_set}")
        else:
            print(f"No records found for RecordSet @id: {record_set}")
    except Exception as e:
        print(f"Error loading records for {record_set}: {e}")

# Display columns for the first available DataFrame
first_rs = record_sets[0] if record_sets else None
if first_rs and first_rs in dataframes:
    print(f"Columns available in DataFrame ({first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping data by key attributes. All references to fields must use their `@id`.


In [ ]:
# EDA: Select a record set with data for analysis
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id}")
    # Find first numeric column (auto-detect if possible)
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]
        print(f"Selected numeric field (by @id): {numeric_field_id}")
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Attempt grouping by another field
        possible_groups = df.columns.difference([numeric_field_id])
        group_field = possible_groups[0] if len(possible_groups) > 0 else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (by @id):")
            display(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a histogram of the numeric field (referenced by its `@id`) if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    if len(df.select_dtypes(include='number').columns) > 0:
        numeric_field_id = df.select_dtypes(include='number').columns[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric field available for visualization.")
else:
    print("No dataframes found for plotting.")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² dataset,
referencing all entities such as record sets and fields via their `@id`. Further analyses can be performed using the dataframes for downstream tasks such as statistical modeling, feature engineering, or machine learning workflows.


> **Key observations**: The dataset provides a rich resource for protein analysis in human extracellular vesicles, ideal for exploratory data analyses and hypothesis generation in proteomics.